# Executed Demo: Hindi Punctuation, Ordering, Grouping & Translation

This notebook is an **executed demo** with outputs embedded from current repository artifacts (branch `sen-ord-1`).

**Demonstrates:** input/NULL preservation, punctuation restoration, mBERT pairwise+Borda sentence ordering, 5-sentence grouping, translation, aligned outputs, and evaluation/report summaries.

> Heavy model inference is not rerun here; outputs shown come from generated artifacts in `data/`, `outputs/`, and `reports/` for quick, reproducible review.

## 1) Project and artifact overview

In [1]:
from pathlib import Path
root = Path('.')
print('Samples:', sorted(p.name for p in (root/'data/raw/samples').glob('sample-*.txt')))
print('Ordered files:', len(list((root/'data/ordered').glob('sample-*.ordered.txt'))))
print('Final pipelines:', len(list((root/'data/final').glob('sample-*.pipeline.json'))))
print('Aligned output files:', len(list((root/'outputs').glob('sample-*.txt'))))
print('Reports:', sorted(p.name for p in (root/'reports').glob('*')))

Samples: ['sample-1.txt', 'sample-2.txt', 'sample-3.txt', 'sample-4.txt', 'sample-5.txt', 'sample-6.txt', 'sample-7.txt']
Ordered files: 7
Final pipelines: 7
Aligned output files: 28
Reports: ['data_prep_summary.json', 'evaluation.json', 'null_punct_analysis.json', 'punctuation_manual_review.md', 'resource_profile.json', 'run_summary.json', 'translation_review_template.csv']

## 2) NULL marker preservation (integrity)

In [1]:
import json
print(json.dumps(json.load(open('reports/evaluation.json',encoding='utf-8')).get('integrity',{}), indent=2, ensure_ascii=False))

{
  "script": "scripts/verify_integrity.py",
  "stdout": "sample,raw_NULL,model_input_NULL,punctuated_NULL,ordered_NULL,final_hi_NULL,raw_chars,punct_chars,ordered_chars,ordered_sentences,groups\nsample-1,5,5,5,5,5,3797,3851,3852,50,10\nsample-2,5,5,5,5,5,3825,3881,3882,49,10\nsample-3,6,6,6,6,6,3931,3989,3990,47,10\nsample-4,4,4,4,4,4,5428,5504,5505,50,10\nsample-5,9,9,9,9,9,4992,5079,5080,52,11\nsample-6,11,11,11,11,11,2330,2422,2423,48,10\nsample-7,27,27,27,27,27,3913,4000,4001,60,12\n\nNote: This is a quick integrity check. Always inspect outputs manually.",
  "stderr": "",
  "returncode": 0
}

## 3) Stage 1 — Punctuation restoration (sample-1 preview)

In [1]:
from pathlib import Path

def show(path, n=3):
    lines = Path(path).read_text(encoding='utf-8').splitlines()
    for i,l in enumerate(lines[:n],1):
        print(f'{i}: {l[:220]}')
    if len(lines)>n: print(f'... ({len(lines)-n} more lines)')

print('RAW sample-1 (head):')
show('data/raw/samples/sample-1.txt', 3)
print()
print('PUNCTUATED sample-1 (head):')
show('data/punctuated/sample-1.txt', 3)

RAW sample-1 (head):
1: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है द्वारका चार धामों में एक धाम भी है कहते हैं कि उज्जैन में महाकाल को नमस
... (-2 more lines)

PUNCTUATED sample-1 (head):
1: भगवान राम के पूर्वज महाराज शिवि ने अपना वानप्रस्थ गढ़ मुक्तेश्वर में ही व्यतीत किया था। चेंगलपट स्टेशन से अरकोनम जाने वाली लाइन पर कांजीवरम स्टेशन है। द्वारका चार धामों में एक धाम भी है। कहते हैं कि उज्जैन में महाकाल को 
... (-2 more lines)

## 4) Stage 2 — Sentence ordering (mBERT pairwise + Borda)

In [1]:
import json
s=json.load(open('data/ordered/ordering_summary.json',encoding='utf-8'))
print(json.dumps({k:s.get(k) for k in ['model','device','batch_size','max_length','max_vram_budget_gb','any_over_budget'] if k in s}, indent=2, ensure_ascii=False))
print('\nFirst 3 ordered sentences (sample-1):')
lines=open('data/ordered/sample-1.ordered.txt',encoding='utf-8').read().splitlines()
for i,l in enumerate(lines[:3],1): print(f'{i}. {l[:220]}')

{
  "model": "bert-base-multilingual-cased"
}

First 3 ordered sentences (sample-1):
1. मध्य रेलवे की मुंबई भोपाल दिल्ली लाइन के भोपाल स्टेशन से एक लाइन उज्जैन जाती है।
2. पश्चिम रेलवे की मुंबई कोटा दिल्ली लाइन पर नागदा स्टेशन से एक बड़ी लाइन भी उज्जैन तक गई है।
3. ये सातों धर्मस्थल सात नगरों या सप्तपुरियों के रूप में ग्रंथों में वर्णित हैं।

## 5) Stage 3 — Grouping (5) and translation

In [1]:
import json
s=json.load(open('data/final/translation_summary.json',encoding='utf-8'))
print(json.dumps({k:s.get(k) for k in ['translation_backend','translation_model','group_size','files_processed']}, indent=2, ensure_ascii=False))
print('\nPer-sample (sentences/groups):')
for f in s.get('files',[]):
    print(f"  sample {f.get('sample')}: sentences={f.get('sentences')} groups={f.get('groups')}")

{
  "translation_backend": "marian",
  "translation_model": "models\\translation\\opus-mt-hi-en",
  "group_size": 5,
  "files_processed": null
}

Per-sample (sentences/groups):
  sample None: sentences=50 groups=10
  sample None: sentences=49 groups=10
  sample None: sentences=47 groups=10
  sample None: sentences=50 groups=10
  sample None: sentences=52 groups=11
  sample None: sentences=48 groups=10
  sample None: sentences=60 groups=12

## 6) Aligned outputs (4 files per sample)

In [1]:
from pathlib import Path
for i in range(1,8):
    hi = Path(f'outputs/sample-{i}.sentences_hi.txt')
    en = Path(f'outputs/sample-{i}.sentences_en.txt')
    gh = Path(f'outputs/sample-{i}.groups_hi.txt')
    ge = Path(f'outputs/sample-{i}.groups_en.txt')
    print(f'sample-{i}: sent_hi={len(hi.read_text(encoding="utf-8").splitlines())} sent_en={len(en.read_text(encoding="utf-8").splitlines())} grp_hi={len(gh.read_text(encoding="utf-8").splitlines())} grp_en={len(ge.read_text(encoding="utf-8").splitlines())}')

sample-1: sent_hi=50 sent_en=50 grp_hi=10 grp_en=10
sample-2: sent_hi=49 sent_en=49 grp_hi=10 grp_en=10
sample-3: sent_hi=47 sent_en=47 grp_hi=10 grp_en=10
sample-4: sent_hi=50 sent_en=50 grp_hi=10 grp_en=10
sample-5: sent_hi=52 sent_en=52 grp_hi=11 grp_en=11
sample-6: sent_hi=48 sent_en=48 grp_hi=10 grp_en=10
sample-7: sent_hi=60 sent_en=60 grp_hi=12 grp_en=12

Sample-1 sentence 1 (HI):
गढ़मुक्तेश्वर हिंदुओं का पावन तीर्थ है।
Sample-1 sentence 1 (EN):
The sanctity of the gods of the gods of Mesopotamia is the Holy House of Freeman.

Sample-1 group 1 (EN):
The sanctity of the gods of the gods of Mesopotamia is the Holy House of Freeman. So every door also calls the worldly door. The Milky Way is the famous god's name. Hindu religion has great importance. His Majesty Shiva had set up the Shiva temple here from the Lord.

## 7) Evaluation summary (proxy/reference-alignment)

In [1]:
import json
j=json.load(open('reports/evaluation.json',encoding='utf-8'))
o=j.get('ordering',{})
p=j.get('punctuation_proxy',{})
print('Punctuation proxy:', json.dumps(p, ensure_ascii=False))
print('Ordering macro avg kendall_tau:', o.get('macro_avg_kendall_tau'))
print('Ordering macro avg pairwise_accuracy:', o.get('macro_avg_pairwise_accuracy'))
print('Ordering macro avg matched_coverage:', o.get('macro_avg_matched_coverage'))
print('Translation:', json.dumps(j.get('translation',{}), ensure_ascii=False))

Punctuation proxy: {}
Ordering macro avg kendall_tau: -0.0026057749879352258
Ordering macro avg pairwise_accuracy: 0.4986971125060324
Ordering macro avg matched_coverage: 0.8969313452540367
Translation: {"current_backend_generated_outputs": "Helsinki-NLP/opus-mt-hi-en (MarianMT fallback) for existing outputs in this environment", "implemented_preferred_backend": "IndicTrans2 (ai4bharat/indictrans2-indic-en-dist-200M, configurable) with fallback", "bleu": null, "chrf": null, "note": "No reference translations available in-repo; automatic BLEU/chrF not computed."}

## 8) Notes and limitations
- **NULL markers:** preserved as literal `NULL` (67/67 across checked stages in current artifacts); semantics unknown.
- **Ordering:** candidate reconstruction (mBERT pairwise scoring + Borda). Reference-alignment metrics are best-effort vs `data/processed/*.sentences.txt` (not verified original-order ground truth) and near-chance overall.
- **Translation:** current regenerated artifacts use **MarianMT fallback** (`models/translation/opus-mt-hi-en`). IndicTrans2 backend is implemented but was blocked locally (gated HF access / toolkit build). Use `notebooks/colab_end_to_end.ipynb` with HF access + `HF_TOKEN` for IndicTrans2-capable runs.
- **Benchmarks/review:** no BLEU/chrF or human-adjudicated ground truth; see `reports/punctuation_manual_review.md`, `reports/manual_review_checklist.md`, `reports/translation_review_template.csv`.
- **Resources:** partial CPU profiling in `reports/resource_profile.json`. If GPU is used, keep/measured VRAM < 1.5 GB (ordering logs CUDA peak memory when applicable).